# 1부 데이터 마이닝과 pandas 복습

Ames 데이터는 미국 아이오와주 Ames시의 2006~2010년 주택 매매 기록 2,930건을 정리한 것이다. 변수는 82개로, 거실 면적·차고 종류·지하실 마감·이웃 동네 같은 *실제로 부동산 가치에 영향을 주는 거의 모든 요인*이 담겨 있다. 타깃 `SalePrice`(매매가)를 예측하는 회귀 문제이며, 캐글 House Prices - Advanced Regression Techniques 경진대회의 정식판(De Cock, 2011)이기도 하다.

트리 모델을 적용하기 전에 데이터를 *손에 익힌다*. 결측치가 어디에 어떻게 분포하는지, 변수 타입이 어떻게 섞여 있는지, 타깃의 분포가 한쪽으로 쏠려 있는지, 어떤 변수가 타깃과 가깝게 움직이는지 — 이 모든 것을 알아야 어떤 트리든 제대로 키울 수 있다.

---

## 1장 데이터 로딩과 첫 탐색

### Ping 1 — URL에서 데이터를 불러와 형태를 확인하기

`pandas`(판다스)의 `read_csv`는 URL을 직접 받는다. 다만 환경에 따라 네트워크가 차단될 수 있으므로, `try/except` 패턴으로 로딩 실패에 대비한다. Colab·Jupyter·로컬 Python·Pyodide 어디서나 같은 코드가 작동하도록 만드는 표준이다.

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"

try:
    df = pd.read_csv(URL)
    print("URL 로딩 성공")
except Exception:
    # 폴백: 핵심 컬럼만 가진 합성 데이터
    rng = np.random.default_rng(42)
    n = 2930
    df = pd.DataFrame({
        "Order": np.arange(1, n+1),
        "Lot Area": (rng.gamma(2.0, 5000, n) + 1500).astype(int),
        "Year Built": rng.integers(1900, 2011, n),
        "Gr Liv Area": (rng.gamma(2.5, 600, n) + 400).astype(int),
        "Overall Qual": rng.integers(1, 11, n),
        "Neighborhood": rng.choice(["NAmes","CollgCr","OldTown","Edwards","Somerst"], n),
        "SalePrice": (rng.gamma(2.0, 80000, n) + 60000).astype(int),
    })
    print("폴백 합성 데이터 사용")

print(f"\n행: {df.shape[0]:,}개")
print(f"열: {df.shape[1]:,}개")
df.head()

#### ✎ 개념 확인

`df.shape`은 `(행 개수, 열 개수)` 형식의 **튜플**(tuple)을 반환한다. Ames 데이터의 경우 `(2930, 82)`인데, 이는 *2,930건의 주택 거래*가 행으로, *82개 변수*가 열로 들어 있음을 의미한다. 행은 한 채의 집을, 열은 그 집을 묘사하는 한 가지 특징을 나타낸다.

머신러닝 용어로 옮기면 행은 **샘플**(sample), 열은 **피처**(feature)다. 타깃 `SalePrice`까지 포함하므로 입력 피처는 정확히 81개다.

### Pong 1 — 데이터의 앞뒤를 살펴 흐름 파악하기

데이터의 **앞부분**(head)과 **뒷부분**(tail)을 함께 봐야 *시간 순으로 정렬된 데이터인가, 무작위 섞임인가, 특수값이 끝에 몰려 있는가* 같은 구조 단서를 잡을 수 있다.

In [ ]:
# 처음 ___개 행을 보려면
df.head(___)

# 마지막 ___개 행을 보려면
df.___(5)

<details><summary>▶ 정답 보기</summary>

```python
df.head(10)
df.tail(5)
```

`head(n)`은 위에서 n개, `tail(n)`은 아래에서 n개 행을 반환한다. n을 생략하면 기본값 5다.
</details>

### Ping 2 — `info()`로 변수 타입과 결측 한눈에 보기

`df.info()`는 각 컬럼의 **데이터 타입**(dtype)과 **비결측 개수**(non-null count)를 한 줄씩 출력한다. 결측 진단의 첫 번째 도구다.

In [ ]:
df.info()

#### ✎ 개념 확인

`info()` 결과에서 `Non-Null Count`가 2930이면 그 컬럼은 결측이 *전혀* 없는 것이다. 2773이라면 2930 - 2773 = 157건이 비어 있다. `Dtype`은 셋 중 하나다.

- `int64`: 정수형. 예: `Year Built`, `Lot Area`
- `float64`: 실수형. *정수처럼 보이지만 결측 한 칸만 있어도 float로 바뀐다* — 이는 `NaN`이 실수 전용이기 때문이다.
- `object`: 문자열(범주형). 예: `Neighborhood`, `MS Zoning`

`Year Built`가 `int64`인데 `Garage Yr Blt`(차고 건축연도)가 `float64`인 까닭은 후자에 결측이 있기 때문이다.

### Pong 2 — `describe()`로 수치형 요약 통계 보기

`describe()`는 *수치형 컬럼만* 골라 평균·표준편차·사분위수를 한 표로 정리한다. 이상치 단서를 빠르게 잡는 도구다.

In [ ]:
# 수치형 컬럼의 요약 통계
df.____()

# 'SalePrice' 한 컬럼만 따로
df[____].describe()

<details><summary>▶ 정답 보기</summary>

```python
df.describe()
df["SalePrice"].describe()
```

`describe()`는 기본적으로 수치형만 본다. 범주형까지 보려면 `df.describe(include="object")`를 쓴다. 중앙값(50%)이 평균과 크게 다르면 *분포가 한쪽으로 쏠려 있다*는 신호다 — `SalePrice`에서 평균 약 18만 달러, 중앙값 약 16만 달러라면 오른쪽 꼬리(고가 주택)가 평균을 끌어올린 것이다.
</details>

---

## 2장 변수 타입의 두 세계 — 수치형과 범주형

### Ping 3 — `select_dtypes`로 수치형과 범주형을 분리하기

머신러닝에서 두 타입은 다른 도구로 처리한다. 수치형은 그대로 모델에 들어가지만, 범주형은 *원-핫 인코딩이나 임베딩*을 거쳐야 한다. 따라서 두 타입을 깨끗이 분리하는 것이 전처리의 출발점이다.

In [ ]:
# 수치형 컬럼만 추출
num_df = df.select_dtypes(include="number")
print(f"수치형 컬럼: {num_df.shape[1]}개")

# 범주형(문자열) 컬럼만 추출
cat_df = df.select_dtypes(include="object")
print(f"범주형 컬럼: {cat_df.shape[1]}개")

# 합계가 전체 컬럼 수와 같은지 확인 (실수 방지)
print(f"\n합계: {num_df.shape[1] + cat_df.shape[1]}개 (전체 {df.shape[1]}개)")

#### ✎ 개념 확인

Ames의 82개 컬럼은 *수치형 39개*와 *범주형 43개*로 거의 반반씩 갈린다. 캘리포니아 주택 데이터(전부 수치형 8개)와의 결정적 차이가 여기 있다.

흥미로운 예외가 있다. `MS SubClass`(주택 유형 코드)는 *숫자로 표기되지만 의미는 범주형*이다. 20=1층 새 집, 60=2층 새 집 식의 코드일 뿐 *60이 20의 세 배가 아니다*. 이런 변수는 모델에 넣기 전 `astype(str)`로 강제 변환해야 한다 — 이 처리는 5장에서 다룬다.

### Pong 3 — 각 타입의 컬럼 이름 목록 출력하기

In [ ]:
# 수치형 컬럼 이름 목록
num_cols = df.select_dtypes(include=____).columns.tolist()
print("수치형:", num_cols[:5], "...")

# 범주형 컬럼 이름 목록
cat_cols = df.select_dtypes(include=____).columns.tolist()
print("범주형:", cat_cols[:5], "...")

<details><summary>▶ 정답 보기</summary>

```python
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()
```

`include="number"`는 `int`와 `float`을 모두 포함하는 약어다. `np.number`로도 같다. `.columns`는 Index 객체를, `.tolist()`는 그것을 일반 리스트로 변환한다.
</details>

---

## 3장 타깃 변수 SalePrice의 분포

### Ping 4 — 히스토그램으로 분포의 모양 보기

회귀 문제에서 *타깃의 분포를 모르면 모든 후속 결정이 불안정하다*. 분포가 한쪽으로 쏠리면 평균은 신뢰할 수 없는 대표값이 되고, 이상치는 모델 학습을 망친다. 첫 시각화는 항상 타깃의 히스토그램이다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 원본 분포
axes[0].hist(df["SalePrice"], bins=50, color="#C0392B", edgecolor="white")
axes[0].set_title("SalePrice 원본 분포", fontsize=13)
axes[0].set_xlabel("매매가 (달러)")
axes[0].set_ylabel("거래 건수")
axes[0].ticklabel_format(style="plain", axis="x")

# 박스플롯으로 이상치 위치 확인
axes[1].boxplot(df["SalePrice"], vert=False, patch_artist=True,
                boxprops=dict(facecolor="#FAF6F1", edgecolor="#1F3A5F"))
axes[1].set_title("SalePrice 박스플롯", fontsize=13)
axes[1].set_xlabel("매매가 (달러)")
axes[1].ticklabel_format(style="plain", axis="x")

plt.tight_layout()
plt.show()

print(f"평균:   ${df['SalePrice'].mean():>10,.0f}")
print(f"중앙값: ${df['SalePrice'].median():>10,.0f}")
print(f"왜도:    {df['SalePrice'].skew():>10.3f}")

#### ✎ 개념 확인

**왜도**(skewness)는 분포의 *비대칭 정도*를 한 숫자로 요약한다. 0이면 좌우 대칭, 양수면 오른쪽 꼬리가 길고(=고가 주택이 평균을 끌어올림), 음수면 왼쪽 꼬리가 길다. Ames의 `SalePrice` 왜도는 약 1.74로, 분포가 *상당히 오른쪽으로 쏠려 있다*.

이 비대칭은 회귀에서 두 가지 문제를 일으킨다. 첫째, 선형회귀의 *잔차 정규성 가정*이 깨진다. 둘째, 트리 모델조차 *극단값 한 채에 분할 기준이 끌려간다*. 해결책은 다음 Ping에서 다룬다.

### Pong 4 — 평균과 중앙값의 차이로 비대칭 진단하기

분포가 대칭이면 평균과 중앙값이 거의 같고, 한쪽으로 쏠리면 두 값이 벌어진다. 이 *간격*이 비대칭의 직관적 지표다.

In [ ]:
mean_price = df["SalePrice"].____()
median_price = df["SalePrice"].____()
gap = mean_price - median_price

print(f"평균 - 중앙값 = ${gap:,.0f}")
print(f"평균 대비 비율: {gap / mean_price * 100:.1f}%")

<details><summary>▶ 정답 보기</summary>

```python
mean_price = df["SalePrice"].mean()
median_price = df["SalePrice"].median()
```

Ames에서 차이가 약 1만 9천 달러, 평균 대비 10% 이상이다. 이 *벌어짐*이 왜도 양수의 또 다른 얼굴이다.
</details>

### Ping 5 — `log1p` 변환으로 분포 정규화하기

오른쪽으로 쏠린 분포에 `log`를 씌우면 큰 값들이 *압축*되어 분포가 종 모양에 가까워진다. `np.log1p(x) = log(1+x)`는 *0이 들어와도 안전한* 로그 변환이다(가격이 0인 행이 있어도 `-inf`가 나오지 않는다).

캐글 House Prices 대회의 평가 지표가 *log(SalePrice)에 대한 RMSE*인 까닭이기도 하다 — 비싼 집의 큰 절대오차와 싼 집의 작은 절대오차를 *같은 비율 오차로* 다루기 위해서다.

In [ ]:
log_price = np.log1p(df["SalePrice"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["SalePrice"], bins=50, color="#C0392B", edgecolor="white")
axes[0].set_title(f"원본 (왜도 = {df['SalePrice'].skew():.2f})", fontsize=13)
axes[0].set_xlabel("매매가 (달러)")
axes[0].ticklabel_format(style="plain", axis="x")

axes[1].hist(log_price, bins=50, color="#1F3A5F", edgecolor="white")
axes[1].set_title(f"log1p 변환 후 (왜도 = {log_price.skew():.2f})", fontsize=13)
axes[1].set_xlabel("log(SalePrice + 1)")

plt.tight_layout()
plt.show()

#### ✎ 개념 확인

`log1p` 변환 후 왜도가 약 0.12까지 떨어진다 — 거의 완벽한 종 모양이다. 트리 모델에는 이 변환이 *반드시 필요하지는 않지만*(트리는 단조 변환에 불변), 선형회귀나 신경망에는 *결정적인 차이*를 만든다.

딥러닝과의 연결: 신경망의 **배치 정규화**(Batch Normalization)도 비슷한 동기에서 만들어졌다 — *각 층의 출력 분포를 안정화*하면 학습이 잘 된다는 직관이다. `log1p`는 그 가장 단순한 형태다.

### Pong 5 — 변환 전후 표준편차 비율 계산하기

분포 압축의 효과를 *상대 표준편차*(평균 대비 표준편차)로 확인한다.

In [ ]:
# 변환 전
cv_before = df["SalePrice"].____() / df["SalePrice"].____()

# 변환 후
log_price = np.____(df["SalePrice"])
cv_after = log_price.std() / log_price.mean()

print(f"변환 전 변동계수: {cv_before:.3f}")
print(f"변환 후 변동계수: {cv_after:.3f}")
print(f"압축 비율: {cv_before / cv_after:.1f}배")

<details><summary>▶ 정답 보기</summary>

```python
cv_before = df["SalePrice"].std() / df["SalePrice"].mean()
log_price = np.log1p(df["SalePrice"])
```

`변동계수`(coefficient of variation, CV)는 단위에 영향받지 않는 산포 지표다. log 변환 후 CV가 약 10~20배 작아지는 것이 정상이다.
</details>

---

## 4장 결측치 — 함정과 의미

### Ping 6 — `isna().sum()`으로 컬럼별 결측 개수 세기

결측치는 머신러닝의 *조용한 폭탄*이다. 트리는 결측을 그대로 받지만 sklearn 구현체는 받지 않고, 선형회귀는 결측 한 칸에도 학습 자체를 거부한다. 처리 전에 *어디에 얼마나 빠져 있는가*부터 정확히 파악한다.

In [ ]:
# 컬럼별 결측 개수 (0인 컬럼은 제외)
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)

# 비율도 함께
miss_pct = (miss / len(df) * 100).round(1)

result = pd.DataFrame({"결측 개수": miss, "결측 비율(%)": miss_pct})
print(f"결측이 있는 컬럼: {len(result)}개\n")
result.head(15)

#### ✎ 개념 확인

Ames에는 결측을 가진 컬럼이 27개나 있다. 그런데 *결측 비율*을 보면 충격적이다.

- `Pool QC`(수영장 품질): 99.6% 결측
- `Misc Feature`(기타 특징): 96.4% 결측  
- `Alley`(골목): 93.2% 결측
- `Fence`(울타리): 80.5% 결측

99.6% 결측이라는 숫자만 보면 *이 컬럼은 쓸모없다, 버리자*가 자연스러운 반응이다. 그러나 Ames의 결측은 두 종류로 갈린다.

**의미 있는 결측**(structural missing): "수영장이 없다", "골목이 없다", "지하실이 없다"를 NA로 표기한 것. 정보가 빠진 것이 아니라 *"해당 없음"*이 NA로 기록된 것이다. 이 경우 단순 dropna는 *데이터를 거의 통째로 버리는* 끔찍한 결과를 낳는다(2,930행 중 2,917행이 Pool QC 결측이므로).

**진짜 누락**(missing at random): `Lot Frontage`(앞마당 너비)의 16.7% 결측은 측정이 안 됐을 뿐이다. 이쪽은 *추정해서 채워야* 한다.

### Pong 6 — 결측을 의미 있는 결측과 진짜 누락으로 분류하기

Ames 데이터셋 문서에 따르면 차고·지하실·수영장·울타리·골목·벽난로 관련 컬럼의 결측은 *"없음"*을 의미한다. 한편 `Lot Frontage`, `Mas Vnr Area`, `Garage Yr Blt`는 측정 누락이다.

In [ ]:
# 결측 비율이 50% 넘는 컬럼 = 구조적 결측 가능성 높음
structural = miss_pct[miss_pct > ____].index.tolist()
print("구조적 결측 후보 (50%+):", structural)

# 결측 비율이 0보다 크고 30% 이하 = 진짜 누락 가능성 높음
real_miss = miss_pct[(miss_pct > 0) & (miss_pct < ____)].index.tolist()
print("\n진짜 누락 후보 (30% 미만):", real_miss[:8])

<details><summary>▶ 정답 보기</summary>

```python
structural = miss_pct[miss_pct > 50].index.tolist()
real_miss = miss_pct[(miss_pct > 0) & (miss_pct < 30)].index.tolist()
```

50%와 30%는 절대 기준이 아니라 *대략적인 경계*다. 실제로는 데이터 문서를 읽고 컬럼 하나하나의 의미를 확인해야 한다.
</details>

### Ping 7 — 의미 있는 결측은 `'None'`으로 채우기

`Pool QC` 같은 컬럼의 NA는 *"수영장 없음"*이라는 *유효한 카테고리*이므로 `'None'` 문자열로 채운다. 이렇게 하면 모델이 *"수영장 없음"과 "수영장 있음·품질 우수"를 구별해서 학습*할 수 있다.

In [ ]:
# 결측이 "없음"을 의미하는 범주형 컬럼들
none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence",
             "Fireplace Qu", "Garage Qual", "Garage Cond",
             "Garage Finish", "Garage Type",
             "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
             "BsmtFin Type 1", "BsmtFin Type 2",
             "Mas Vnr Type"]

# 실제로 데이터에 있는 컬럼만 골라서 채우기
none_cols = [c for c in none_cols if c in df.columns]

df_clean = df.copy()
df_clean[none_cols] = df_clean[none_cols].fillna("None")

# 채우기 전후 비교
print("채우기 전 결측 합계:", df[none_cols].isna().sum().sum())
print("채우기 후 결측 합계:", df_clean[none_cols].isna().sum().sum())

#### ✎ 개념 확인

`fillna("None")`은 NA 자리에 문자열 `"None"`을 넣는다. 파이썬의 `None` 객체가 아니라 *"None"이라는 다섯 글자 문자열*이라는 점이 중요하다. 이렇게 해야 다음 단계의 원-핫 인코딩에서 *"None"이라는 별도의 카테고리*로 잡힌다.

머신러닝 파이프라인 연결: `sklearn.impute.SimpleImputer(strategy="constant", fill_value="None")`이 같은 일을 한다. 본 강의의 트리가족 책 후반부에서는 이런 처리를 `Pipeline`으로 묶어 자동화한다.

### Pong 7 — 진짜 누락은 중앙값으로 채우기

`Lot Frontage`(앞마당 너비)의 결측 490건은 측정 누락이다. 수치형이므로 *해당 동네의 중앙값*으로 채우는 것이 합리적이다 — 같은 동네 집들은 비슷한 앞마당 너비를 가질 것이라는 도메인 지식의 반영이다.

In [ ]:
# 단순 중앙값 채우기
median_lf = df_clean["Lot Frontage"].____()
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].____(median_lf)

# 같은 방법으로 'Mas Vnr Area'도 0으로 채우기 ("벽돌 외장이 없으면 면적도 0")
df_clean["Mas Vnr Area"] = df_clean["Mas Vnr Area"].fillna(____)

# Garage Yr Blt 결측은 "차고 없음"이므로 0으로
df_clean["Garage Yr Blt"] = df_clean["Garage Yr Blt"].fillna(____)

# 남은 결측 확인
print("처리 후 결측 컬럼:", (df_clean.isna().sum() > 0).sum(), "개")

<details><summary>▶ 정답 보기</summary>

```python
median_lf = df_clean["Lot Frontage"].median()
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(median_lf)
df_clean["Mas Vnr Area"] = df_clean["Mas Vnr Area"].fillna(0)
df_clean["Garage Yr Blt"] = df_clean["Garage Yr Blt"].fillna(0)
```

더 정교한 방법으로 *동네별 중앙값*을 쓸 수 있다: `df.groupby("Neighborhood")["Lot Frontage"].transform("median")`. 7장의 groupby에서 다룬다.
</details>

---

## 5장 범주형 변수의 다양한 얼굴

### Ping 8 — `value_counts`로 범주의 분포 보기

범주형 변수를 모델에 넣기 전 *각 범주가 몇 개씩 있는가*를 반드시 확인한다. 어떤 범주가 한두 건뿐이라면 모델이 그 범주를 학습할 수 없고, 한 범주가 95% 이상을 차지하면 *그 변수는 정보가 거의 없는 셈*이다.

In [ ]:
# Neighborhood: 동네별 거래 건수
print("=== Neighborhood (동네) 상위 10개 ===")
print(df_clean["Neighborhood"].value_counts().head(10))

print("\n=== Overall Qual (전반적 품질, 1~10) ===")
print(df_clean["Overall Qual"].value_counts().sort_index())

print(f"\n동네 종류 수: {df_clean['Neighborhood'].nunique()}개")

#### ✎ 개념 확인

Ames의 `Neighborhood`(동네)는 28개 범주를 가지며 `NAmes`(North Ames, 약 15%)가 가장 흔하고, `GrnHill`(Green Hills)은 단 2건뿐이다. 이런 *희소 범주*(rare category)는 트리 모델에서 *과적합의 원천*이 된다 — 단 한 채의 주택이 분할 기준이 되면 일반화가 무너진다.

해결책은 두 가지다. 첫째, 희소 범주를 `"Other"`로 묶기. 둘째, 트리 모델의 `min_samples_leaf`로 한 잎에 최소 몇 개 샘플이 있어야 하는지 강제하기. 본 책 후반부에서 다룬다.

### Pong 8 — 가장 흔한 동네와 가장 흔한 외장재 찾기

In [ ]:
# 가장 흔한 동네
top_neighborhood = df_clean["Neighborhood"].____().____()
print(f"가장 흔한 동네: {top_neighborhood}")

# 가장 흔한 외장재 (Exterior 1st)
top_exterior = df_clean["Exterior 1st"].value_counts().____()
print(f"가장 흔한 외장재: {top_exterior}")

<details><summary>▶ 정답 보기</summary>

```python
top_neighborhood = df_clean["Neighborhood"].value_counts().idxmax()
top_exterior = df_clean["Exterior 1st"].value_counts().idxmax()
```

`idxmax()`는 *최댓값을 가진 인덱스*를 반환한다. `mode()`로도 같은 결과를 얻지만 `idxmax`가 더 직관적이다.
</details>

### Ping 9 — 원-핫 인코딩으로 범주형을 수치형으로 펼치기

대부분의 머신러닝 모델은 문자열 `"NAmes"`를 직접 받지 못한다. **원-핫 인코딩**(one-hot encoding)은 *한 범주형 컬럼을 여러 0/1 컬럼으로 펼치는* 변환이다. 동네가 28개 범주라면 28개의 0/1 컬럼이 생긴다 — 한 행에서 *정확히 하나의 컬럼만 1*, 나머지는 0이다.

In [ ]:
# 간단한 예시: 동네 컬럼 하나만
example = pd.get_dummies(df_clean[["Neighborhood"]].head(5), prefix="N")
example

#### ✎ 개념 확인

원-핫 인코딩은 *직교 단위벡터*(orthogonal unit vector)의 데이터 버전이다. 28차원 공간에서 NAmes는 (1,0,0,...,0), CollgCr는 (0,1,0,...,0)으로 표현된다. 이 표현은 *모든 범주가 서로 같은 거리*에 있다는 가정을 깐다 — 즉 NAmes와 CollgCr의 거리가 NAmes와 GrnHill의 거리와 같다.

딥러닝과의 연결: **임베딩 레이어**(embedding layer)는 원-핫의 한계를 극복한 변환이다. 28차원 0/1 대신 *학습 가능한 저차원 실수 벡터*(예: 8차원)로 매핑하여, 비슷한 동네는 비슷한 벡터를 갖도록 한다. `nn.Embedding(28, 8)`의 직관이 여기서 출발한다.

### Pong 9 — 전체 범주형 컬럼을 한 번에 원-핫 인코딩하기

In [ ]:
# 모든 범주형 컬럼 자동 추출
cat_cols = df_clean.select_dtypes(include=____).columns.tolist()
print(f"인코딩할 범주형 컬럼: {len(cat_cols)}개")

# 한 번에 변환
df_encoded = pd.get_dummies(df_clean, columns=____, drop_first=False)

print(f"\n변환 전: {df_clean.shape[1]}개 컬럼")
print(f"변환 후: {df_encoded.shape[1]}개 컬럼")
print(f"늘어난 컬럼: {df_encoded.shape[1] - df_clean.shape[1]}개")

<details><summary>▶ 정답 보기</summary>

```python
cat_cols = df_clean.select_dtypes(include="object").columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=False)
```

82개 컬럼이 약 280여 개로 부풀어 오른다 — 범주가 많은 변수일수록 더 많은 컬럼을 만든다. `drop_first=True`로 두면 각 변수당 한 컬럼씩 줄여 *완벽한 다중공선성*을 피할 수 있다(선형 모델에서 중요, 트리에는 영향 없음).
</details>

---

## 6장 변수 간 관계와 상관계수

### Ping 10 — 상관계수 행렬로 변수 가족 찾기

머신러닝에서 *서로 비슷하게 움직이는 변수*들을 발견하는 것은 두 가지로 중요하다. 첫째, *타깃과 강하게 상관된 변수*는 좋은 예측자가 된다. 둘째, *서로 강하게 상관된 입력 변수들*은 회귀에서 다중공선성을 일으키고 트리에서는 변수 중요도를 분산시킨다.

In [ ]:
# 수치형 컬럼만 골라서 상관계수 계산
num_cols = df_clean.select_dtypes(include="number").columns.tolist()
corr = df_clean[num_cols].corr()

# SalePrice와의 상관계수 절댓값 상위 15개
target_corr = corr["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
print("SalePrice와 가장 강하게 상관된 변수 15개:")
print(target_corr.head(15).round(3))

#### ✎ 개념 확인

`Overall Qual`(전반적 품질, 0.80), `Gr Liv Area`(거실 면적, 0.71), `Garage Cars`(차고 차량 수용, 0.65), `Garage Area`(차고 면적, 0.64)가 상위권을 차지한다. *품질 점수 하나가 면적보다 강한 예측력을 가진다*는 사실은 도메인적으로도 흥미롭다 — 부동산에서 "리모델링 상태"는 면적만큼 가격을 좌우한다는 뜻이다.

`Garage Cars`와 `Garage Area`가 둘 다 상위권에 있는 것이 *사촌처럼 닮은 변수*의 전형적 사례다. 둘은 사실상 같은 정보(차고 크기)를 다른 단위로 표현한다.

### Pong 10 — 입력 변수끼리의 다중공선성 발견하기

`Garage Cars`와 `Garage Area`의 상관계수를 직접 계산해 *사촌 변수*임을 확인한다.

In [ ]:
# 두 변수의 상관계수
corr_garage = df_clean["Garage Cars"].____(df_clean["Garage Area"])
print(f"Garage Cars vs Garage Area 상관계수: {corr_garage:.3f}")

# 0.8 넘는 변수쌍 모두 찾기
high_corr = []
for i, c1 in enumerate(num_cols):
    for c2 in num_cols[i+1:]:
        r = df_clean[c1].corr(df_clean[c2])
        if abs(r) > ____ and c1 != "SalePrice" and c2 != "SalePrice":
            high_corr.append((c1, c2, r))

print(f"\n상관계수 |r| > 0.8 인 변수쌍: {len(high_corr)}개")
for c1, c2, r in high_corr[:5]:
    print(f"  {c1:20s}  ↔  {c2:20s}  r = {r:.3f}")

<details><summary>▶ 정답 보기</summary>

```python
corr_garage = df_clean["Garage Cars"].corr(df_clean["Garage Area"])
if abs(r) > 0.8 and ...:
```

`Garage Cars` ↔ `Garage Area`(0.89), `Gr Liv Area` ↔ `TotRms AbvGrd`(0.81), `1st Flr SF` ↔ `Total Bsmt SF`(0.80) 등이 잡힌다. *면적 가족*과 *차고 가족*이 각각 사촌 무리를 이루고 있음을 알 수 있다.
</details>

### Ping 11 — 산점도로 이상치 시각화하기

상관계수는 한 숫자로 관계를 요약하지만, *어디서 관계가 깨지는가*는 산점도로만 보인다. `Gr Liv Area`(거실 면적) vs `SalePrice` 산점도는 Ames의 *유명한 이상치 4채*를 드러낸다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df_clean["Gr Liv Area"], df_clean["SalePrice"],
           alpha=0.4, s=20, color="#1F3A5F")

# 거실 면적 4000 sq ft 넘는 이상치 강조
outliers = df_clean[df_clean["Gr Liv Area"] > 4000]
ax.scatter(outliers["Gr Liv Area"], outliers["SalePrice"],
           s=80, facecolors="none", edgecolors="#C0392B", linewidths=2,
           label=f"이상치 후보 ({len(outliers)}개)")

ax.set_xlabel("Gr Liv Area (거실 면적, sq ft)", fontsize=12)
ax.set_ylabel("SalePrice (매매가, 달러)", fontsize=12)
ax.set_title("거실 면적과 매매가의 관계 — 이상치 발견", fontsize=13)
ax.ticklabel_format(style="plain", axis="y")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print("\n이상치 4채:")
print(outliers[["Gr Liv Area", "SalePrice", "Overall Qual", "Sale Condition"]])

#### ✎ 개념 확인

데이터셋 원작자 De Cock(2011)은 *거실 면적 4,000 sq ft 이상의 거래 4건을 제거할 것*을 명시적으로 권한다. 그중 두 채는 *부분 판매*(Partial sale, 미완공 상태로 매매)였고, 다른 두 채는 *비정상적으로 큰 저가 거래*다. 이 4건은 모든 머신러닝 모델, 특히 트리에서 학습을 일그러뜨린다.

트리가 어떻게 일그러지는가: 부스팅의 첫 트리가 이 극단값을 *큰 잔차*로 본 다음, 두 번째 트리가 그 잔차에 집중해 *4건의 극단값에만 맞춘 분할*을 만든다. 이는 부스팅이 이상치에 민감한 핵심 원리이며, 4부 AdaBoost와 5부 GBM에서 자세히 다룬다.

### Pong 11 — 이상치를 제거한 데이터 만들기

In [ ]:
# 거실 면적 4000 미만만 남기기
df_no_outlier = df_clean[df_clean["Gr Liv Area"] < ____].copy()
print(f"제거 전: {len(df_clean):,}행")
print(f"제거 후: {len(df_no_outlier):,}행")
print(f"제거된 행: {len(df_clean) - len(df_no_outlier)}행")

# 상관계수가 어떻게 변하는가?
r_before = df_clean["Gr Liv Area"].corr(df_clean["SalePrice"])
r_after = df_no_outlier["Gr Liv Area"].corr(df_no_outlier["____"])
print(f"\n상관계수 변화: {r_before:.3f}  →  {r_after:.3f}")

<details><summary>▶ 정답 보기</summary>

```python
df_no_outlier = df_clean[df_clean["Gr Liv Area"] < 4000].copy()
r_after = df_no_outlier["Gr Liv Area"].corr(df_no_outlier["SalePrice"])
```

이상치 4건만 제거해도 거실 면적과 매매가의 상관계수가 약 0.71 → 0.74로 오른다. *전체 2,930개 중 4개의 영향력*이 이만큼이다.
</details>

> 이전 Pong을 풀지 않았어도 다음 절이 작동하도록, 핵심 데이터프레임을 보장하는 셀이다. 이미 Pong을 풀었다면 그대로 두고 실행해도 같은 결과가 나온다.

In [ ]:
# 이전 Pong에서 만들지 못한 경우를 대비해 df_no_outlier를 보장
if "df_no_outlier" not in dir():
    df_no_outlier = df_clean[df_clean["Gr Liv Area"] < 4000].copy()
print(f"df_no_outlier: {df_no_outlier.shape}")

---

## 7장 `groupby`로 집단별 패턴 발견하기

### Ping 12 — 동네별 평균 매매가 계산하기

`groupby`는 *같은 범주의 행들을 묶어 집단별 통계를 계산*하는 pandas의 핵심 도구다. 부동산 분석에서 "동네별 평균 가격"은 입지(location)의 가치를 한 표로 압축한다.

In [ ]:
# 동네별 평균 매매가
price_by_nbhd = (df_no_outlier
                 .groupby("Neighborhood")["SalePrice"]
                 .agg(["mean", "median", "count"])
                 .round(0)
                 .sort_values("median", ascending=False))

print("동네별 매매가 (중앙값 기준 정렬, 상위 10개):")
print(price_by_nbhd.head(10))
print("\n동네별 매매가 (하위 5개):")
print(price_by_nbhd.tail(5))

#### ✎ 개념 확인

`NoRidge`(Northridge Heights), `NridgHt`(Northridge), `StoneBr`(Stone Brook)가 평균 30만 달러를 넘는 *프리미엄 동네*이고, `MeadowV`(Meadow Village), `IDOTRR`(Iowa DOT and Rail Road)이 평균 10만 달러대의 저가 동네다. *같은 도시에서 동네에 따라 평균 가격이 3배 차이*가 난다 — 입지의 위력이다.

이 통계는 곧 *결측치 채우기에도 쓸 수 있다*. `Lot Frontage` 결측을 *전체 중앙값*이 아니라 *해당 동네의 중앙값*으로 채우면 더 정확하다. 다음 Pong에서 실습한다.

### Pong 12 — 동네별 중앙값으로 결측 채우기

In [ ]:
# 동네별 Lot Frontage 중앙값을 같은 행의 결측 자리에 넣기
# transform은 그룹 통계를 원본 길이로 펼쳐서 반환한다
nbhd_median_lf = df_no_outlier.groupby("Neighborhood")["Lot Frontage"].____("median")

# fillna에 Series를 주면 인덱스 매칭으로 채워진다
df_no_outlier["Lot Frontage"] = df_no_outlier["Lot Frontage"].____(nbhd_median_lf)

print("Lot Frontage 결측 처리 후:", df_no_outlier["Lot Frontage"].isna().sum(), "개")

<details><summary>▶ 정답 보기</summary>

```python
nbhd_median_lf = df_no_outlier.groupby("Neighborhood")["Lot Frontage"].transform("median")
df_no_outlier["Lot Frontage"] = df_no_outlier["Lot Frontage"].fillna(nbhd_median_lf)
```

`transform`은 그룹 통계를 *원래 행 길이로 다시 펼쳐* 반환한다. `agg`나 `mean`은 그룹 수만큼 짧은 결과를, `transform`은 원본과 같은 길이의 결과를 준다. fillna에 같은 인덱스의 Series를 주면 인덱스가 일치하는 자리에 결측을 채운다.
</details>

### Ping 13 — 품질 등급별 평균 가격으로 비선형 관계 보기

`Overall Qual`(전반적 품질, 1~10 점수)과 가격의 관계는 *비선형*이다 — 9점에서 10점으로 한 단계 오를 때 가격이 *몇 배로 뛴다*. 이는 트리 모델이 빛을 발하는 전형적 패턴이다.

In [ ]:
import matplotlib.pyplot as plt

price_by_qual = df_no_outlier.groupby("Overall Qual")["SalePrice"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(price_by_qual.index, price_by_qual["mean"],
              color="#1F3A5F", edgecolor="white", width=0.7)

# 각 막대 위에 거래 건수 표시
for i, (idx, row) in enumerate(price_by_qual.iterrows()):
    ax.text(idx, row["mean"] + 5000, f"n={int(row['count'])}",
            ha="center", fontsize=9, color="#666")

ax.set_xlabel("Overall Qual (전반적 품질 점수)", fontsize=12)
ax.set_ylabel("평균 매매가 (달러)", fontsize=12)
ax.set_title("품질 등급별 평균 매매가 — 9→10에서 가격이 폭증", fontsize=13)
ax.ticklabel_format(style="plain", axis="y")
ax.set_xticks(range(1, 11))
plt.tight_layout()
plt.show()

# 9점 대비 10점 가격 비율
ratio = price_by_qual.loc[10, "mean"] / price_by_qual.loc[9, "mean"]
print(f"\n품질 9점 평균: ${price_by_qual.loc[9, 'mean']:,.0f}")
print(f"품질 10점 평균: ${price_by_qual.loc[10, 'mean']:,.0f}")
print(f"비율: {ratio:.2f}배")

#### ✎ 개념 확인

품질 점수가 1단계 오를 때 가격은 *일정한 금액*이 아니라 *일정한 비율*로 오르는 것에 가깝다. 1→2는 5만 달러 차이지만 9→10은 *15만 달러 차이*다. 이 비선형성을 선형회귀는 한 직선으로 잡지 못하고, 단일 결정 트리도 잎의 개수가 부족하면 잡지 못한다.

부스팅의 위력이 여기서 나온다 — *첫 트리가 잡지 못한 비선형 잔차를 두 번째 트리가 더 깊은 분할로 잡아내는* 식으로 누적되기 때문이다. 5부 GBM에서 잔차 학습의 단계를 시각화한다.

### Pong 13 — 동네 × 품질의 교차 평균 만들기

`groupby`에 두 변수를 넘기면 *교차표*가 만들어진다. 이는 *변수 간 상호작용*의 신호다.

In [ ]:
# 동네 × 품질의 평균 가격 교차표 (상위 5개 동네만)
top5_nbhd = df_no_outlier["Neighborhood"].value_counts().head(5).____.tolist()
sub = df_no_outlier[df_no_outlier["Neighborhood"].isin(top5_nbhd)]

cross = sub.groupby([____, ____])["SalePrice"].mean().unstack().round(0)
print("동네 × 품질의 평균 매매가:")
cross

<details><summary>▶ 정답 보기</summary>

```python
top5_nbhd = df_no_outlier["Neighborhood"].value_counts().head(5).index.tolist()
cross = sub.groupby(["Neighborhood", "Overall Qual"])["SalePrice"].mean().unstack().round(0)
```

`value_counts().head(5).index`로 상위 5개 동네 이름을 얻고, `isin`으로 필터링한다. `unstack`은 멀티인덱스의 안쪽 레벨을 *컬럼*으로 펼쳐 교차표 모양을 만든다.
</details>

---

## 8장 특성공학 — 새 변수 만들기

### Ping 14 — 총면적 변수 만들기

Ames에는 면적 변수가 흩어져 있다 — 1층(`1st Flr SF`), 2층(`2nd Flr SF`), 지하실(`Total Bsmt SF`), 거실(`Gr Liv Area`). 도메인 지식상 *총거주면적*이 가격의 핵심 결정요인이다. 흩어진 변수를 합쳐 *하나의 강력한 변수*로 만든다.

In [ ]:
# 총면적 = 1층 + 2층 + 지하실
df_no_outlier["Total SF"] = (df_no_outlier["1st Flr SF"] 
                              + df_no_outlier["2nd Flr SF"] 
                              + df_no_outlier["Total Bsmt SF"])

# 화장실 총 개수 = 전체 욕실(1.0점) + 절반 욕실(0.5점)
df_no_outlier["Total Bath"] = (df_no_outlier["Full Bath"]
                                + 0.5 * df_no_outlier["Half Bath"]
                                + df_no_outlier["Bsmt Full Bath"]
                                + 0.5 * df_no_outlier["Bsmt Half Bath"])

# 새 변수와 SalePrice의 상관계수
new_corr = df_no_outlier[["Total SF", "Total Bath", "Gr Liv Area", "SalePrice"]].corr()
print("새 변수와 SalePrice의 상관계수:")
print(new_corr["SalePrice"].round(3))

#### ✎ 개념 확인

`Total SF`(총면적)는 `Gr Liv Area`(거실 면적, r=0.71)보다 *더 강한 상관계수*(r≈0.78)를 보인다. 흩어져 있던 정보를 *도메인 지식으로 합치는 것*이 특성공학의 본질이다.

이 변수가 모델 성능에 기여할까? 트리는 두 변수의 합을 *자동으로 학습하지는 못한다*. `if 1stFlrSF > a AND 2ndFlrSF > b AND TotalBsmtSF > c` 같은 분할 3개를 거쳐야 비슷한 효과를 내므로, *미리 합쳐 주는 것이 트리에게 한 분할로 끝낼 기회를 준다*. 다만 깊은 부스팅 트리는 결국 비슷한 효과에 도달하므로, 특성공학의 효과는 *모델이 단순할수록 크다*.

### Pong 14 — 주택 연령과 리모델링 연수 만들기

`Year Built`(건축 연도), `Year Remod/Add`(리모델링 연도), `Yr Sold`(매매 연도)에서 *매매 시점의 주택 연령*과 *리모델링 후 경과 연수*를 계산한다.

In [ ]:
# 매매 시점 기준 주택 연령
df_no_outlier["House Age"] = df_no_outlier["Yr Sold"] - df_no_outlier["____"]

# 리모델링 후 경과 연수 (리모델링 없는 집은 건축 연도와 같음)
df_no_outlier["Years Since Remod"] = df_no_outlier["Yr Sold"] - df_no_outlier["Year Remod/Add"]

# 음수가 나오면 데이터 오류 — 0으로 처리
df_no_outlier["House Age"] = df_no_outlier["House Age"].clip(lower=____)
df_no_outlier["Years Since Remod"] = df_no_outlier["Years Since Remod"].clip(lower=0)

print("새 변수 요약:")
print(df_no_outlier[["House Age", "Years Since Remod"]].describe().round(1))

<details><summary>▶ 정답 보기</summary>

```python
df_no_outlier["House Age"] = df_no_outlier["Yr Sold"] - df_no_outlier["Year Built"]
df_no_outlier["House Age"] = df_no_outlier["House Age"].clip(lower=0)
```

`clip(lower=0)`은 0보다 작은 값을 모두 0으로 자른다. Ames에는 *건축 연도가 매매 연도보다 뒤*인 *데이터 오류 1건*이 있어 이 처리가 필요하다.
</details>

### Ping 15 — 비율 변수와 상호작용 변수 만들기

*평당 가격*, *방당 면적*, *욕실당 침실 수* 같은 *비율 변수*는 절대값이 가리지 못하는 패턴을 드러낸다.

In [ ]:
# 방당 면적
df_no_outlier["SF Per Room"] = df_no_outlier["Gr Liv Area"] / df_no_outlier["TotRms AbvGrd"]

# 침실 대 욕실 비율
df_no_outlier["Bed Bath Ratio"] = (df_no_outlier["Bedroom AbvGr"] 
                                    / (df_no_outlier["Total Bath"] + 0.5))

# 차고 면적당 차량 수용 (효율성)
df_no_outlier["Garage Density"] = np.where(
    df_no_outlier["Garage Area"] > 0,
    df_no_outlier["Garage Cars"] / df_no_outlier["Garage Area"] * 100,
    0
)

# 새 변수 3개의 상관계수
print("비율 변수와 SalePrice의 상관계수:")
for col in ["SF Per Room", "Bed Bath Ratio", "Garage Density"]:
    r = df_no_outlier[col].corr(df_no_outlier["SalePrice"])
    print(f"  {col:18s}  r = {r:+.3f}")

#### ✎ 개념 확인

`SF Per Room`(방당 면적)이 *양의 상관*을 보이면 *큰 방 = 비싼 집*이라는 직관이 사실임을 뜻하고, `Bed Bath Ratio`(침실/욕실)가 *음의 상관*을 보이면 *욕실 대비 침실이 많은 집은 저가*라는 신호다(욕실이 부족한 다세대 구조).

이런 변수들은 *비선형 상호작용*을 *선형 형태로 변환*한 결과다. 신경망이 *자동으로* 학습하는 패턴을 *손으로 짚어주는* 작업이 곧 특성공학이다. 이 손작업과 자동 학습의 경계가 모호해진 것이 딥러닝 시대의 변화이지만, *작은 데이터에서는 여전히 손작업이 압도적으로 유리하다* — Ames(2,930행)는 정확히 그 영역이다.

### Pong 15 — 모든 처리를 하나의 함수로 묶기

지금까지의 전처리 단계를 *재사용 가능한 함수*로 정리한다. 머신러닝 프로젝트의 *프로덕션 코드*는 이런 모습이어야 한다.

In [ ]:
def prepare_ames(df_raw):
    """Ames 원본 데이터를 받아 전처리된 DataFrame을 반환한다."""
    df = df_raw.copy()
    
    # 1) 식별자 제거
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    
    # 2) 의미 있는 결측을 'None'으로
    none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
                 "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
                 "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
                 "BsmtFin Type 1", "BsmtFin Type 2", "Mas Vnr Type"]
    for c in none_cols:
        if c in df.columns:
            df[c] = df[c].____("None")
    
    # 3) 진짜 누락 처리
    if "Lot Frontage" in df.columns:
        lf_med = df.groupby("Neighborhood")["Lot Frontage"].transform("median")
        df["Lot Frontage"] = df["Lot Frontage"].fillna(lf_med).fillna(df["Lot Frontage"].median())
    df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(0)
    df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(0)
    
    # 4) 이상치 제거
    df = df[df["Gr Liv Area"] < ____].copy()
    
    # 5) 특성공학
    df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    df["Total Bath"] = (df["Full Bath"] + 0.5*df["Half Bath"]
                       + df["Bsmt Full Bath"] + 0.5*df["Bsmt Half Bath"])
    df["House Age"] = (df["Yr Sold"] - df["Year Built"]).clip(lower=0)
    df["Years Since Remod"] = (df["Yr Sold"] - df["Year Remod/Add"]).clip(lower=0)
    
    return df

# 함수 적용 (URL에서 원본 다시 로드)
df_raw = pd.read_csv(URL)
df_final = prepare_ames(df_raw)
print(f"원본:    {df_raw.shape}")
print(f"전처리 후: {df_final.shape}")
print(f"남은 결측: {df_final.isna().sum().sum()}개")

<details><summary>▶ 정답 보기</summary>

```python
df[c] = df[c].fillna("None")
df = df[df["Gr Liv Area"] < 4000].copy()
```

이 함수는 트리가족 책 전체에서 *한 줄로 데이터를 준비*하는 표준 도구가 된다. 2부 결정 트리부터 7부 종합 비교까지, 매 부의 첫 셀에서 이 함수를 호출하면 같은 전처리가 보장된다.
</details>

---

## 종합 정리 — 1부에서 손에 익힌 것

이번 부에서 다음 도구들이 *손에 익었다*.

| 도구 | 용도 | Ames에서의 발견 |
|---|---|---|
| `shape`, `head`, `info` | 데이터 구조 파악 | 2,930행 × 82열, 결측 27 컬럼 |
| `describe` | 수치형 요약 | 평균-중앙값 간격으로 비대칭 감지 |
| `select_dtypes` | 타입 분리 | 수치 39 + 범주 43 |
| `hist`, `skew` | 분포 진단 | SalePrice 왜도 1.74 |
| `log1p` | 분포 정규화 | 왜도 1.74 → 0.12 |
| `isna().sum()` | 결측 진단 | Pool QC 99.6% 결측의 정체 |
| `fillna("None")` | 의미 있는 결측 처리 | "수영장 없음"을 카테고리로 |
| `value_counts` | 범주 분포 | 동네 28개, 희소 범주 식별 |
| `get_dummies` | 원-핫 인코딩 | 82 → 280여 컬럼 |
| `corr` | 상관계수 | Overall Qual r=0.80이 최강 |
| `groupby`, `transform` | 집단별 통계 | 동네별 중앙값으로 결측 채우기 |
| 산점도 + 이상치 | 관계의 깨짐 발견 | 거실 4000+ sq ft 이상치 4건 |
| 특성공학 | 새 변수 생성 | Total SF, House Age, Bath Ratio |

다음 부에서는 이렇게 준비된 데이터 위에 *결정 트리 한 그루*를 키운다. 그리고 그 한 그루가 *왜 충분하지 않은가*를 데이터로 확인한 뒤, 3부에서 *수백 그루의 랜덤 포레스트*로 넘어간다.

---

## 유사 연습문제 10선

각 문제의 답안은 바로 아래의 토글에 있다. 직접 풀고 나서 확인하기를 권한다.

### 연습 1. SalePrice의 25% 분위수와 75% 분위수의 차이는?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
q1 = df["SalePrice"].quantile(0.25)
q3 = df["SalePrice"].quantile(0.75)
iqr = q3 - q1
print(f"IQR = ${iqr:,.0f}")  # 약 84,025
```
</details>

### 연습 2. `Overall Qual` 점수가 8 이상인 주택은 몇 채인가?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
n_high = (df["Overall Qual"] >= 8).sum()
print(f"{n_high}채")  # 약 466채
```
</details>

### 연습 3. 결측이 가장 많은 컬럼 3개의 이름을 출력하라.

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
top3_miss = df.isna().sum().sort_values(ascending=False).head(3)
print(top3_miss.index.tolist())  # ['Pool QC', 'Misc Feature', 'Alley']
```
</details>

### 연습 4. `Neighborhood`별 *주택 수*가 가장 많은 동네와 가장 적은 동네를 출력하라.

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
counts = df["Neighborhood"].value_counts()
print(f"최대: {counts.idxmax()} ({counts.max()}채)")
print(f"최소: {counts.idxmin()} ({counts.min()}채)")
# NAmes 443채, GrnHill 2채
```
</details>

### 연습 5. `Year Built`가 1900년 이전인 주택은 몇 채인가?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
old_houses = (df["Year Built"] < 1900).sum()
print(f"{old_houses}채")
```
</details>

### 연습 6. `Gr Liv Area`가 평균보다 큰 주택의 평균 매매가는?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
mean_area = df["Gr Liv Area"].mean()
big_homes = df[df["Gr Liv Area"] > mean_area]
print(f"${big_homes['SalePrice'].mean():,.0f}")
```
</details>

### 연습 7. `MS Zoning` 컬럼의 각 범주별 평균 매매가를 내림차순으로 출력하라.

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
result = df.groupby("MS Zoning")["SalePrice"].mean().sort_values(ascending=False)
print(result.round(0))
```
</details>

### 연습 8. `Garage Cars`가 0인 주택(차고 없음)과 1대 이상인 주택의 평균 가격 차이는?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
no_garage = df[df["Garage Cars"] == 0]["SalePrice"].mean()
with_garage = df[df["Garage Cars"] > 0]["SalePrice"].mean()
print(f"차이: ${with_garage - no_garage:,.0f}")
```
</details>

### 연습 9. SalePrice를 log 변환한 후, 표준편차가 변환 전보다 얼마나 줄었는가?

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
sd_before = df["SalePrice"].std()
sd_after = np.log1p(df["SalePrice"]).std()
print(f"변환 전: {sd_before:.0f}")
print(f"변환 후: {sd_after:.3f}")
print(f"비율: {sd_before / sd_after:.0f}배")
```
</details>

### 연습 10. 새 변수 `Quality Per SF` = `Overall Qual` / `Gr Liv Area` * 1000 을 만들고 `SalePrice`와의 상관계수를 구하라.

In [ ]:
# 여기에 코드 작성

<details><summary>▶ 정답 보기</summary>

```python
df["Quality Per SF"] = df["Overall Qual"] / df["Gr Liv Area"] * 1000
r = df["Quality Per SF"].corr(df["SalePrice"])
print(f"r = {r:.3f}")
# 음의 상관이 나오는데, "단위 면적당 품질이 높을수록 작은 고급 주택"을 의미
```
</details>